# Receive

> Connect to iverson.io and type received glyphs.

Speak a **number** assigned in Sound Names (or a basedpl English name like `plus`).
iverson.io sends digits/names over BLE; `iverson-receive --code XXXX` maps them to glyphs and types them.


In [ ]:
#| default_exp receive


In [ ]:
#| export
import argparse, asyncio, json, logging, sys, time, traceback, unicodedata
from pathlib import Path


In [ ]:
#| export
log = logging.getLogger('iverson.receive')

def _setup_log(verbose=False):
    "Configure stderr logging once."
    if log.handlers: return
    h = logging.StreamHandler(sys.stderr)
    h.setFormatter(logging.Formatter('[%(asctime)s] [iverson-recv] %(levelname)s %(message)s', '%H:%M:%S'))
    log.addHandler(h)
    log.setLevel(logging.DEBUG if verbose else logging.INFO)
    log.propagate = False


In [ ]:
#| export
try:
    from bleak import BleakClient, BleakScanner
    _bleak = True
except Exception:
    _bleak = False

try:
    from iverson.bt import SERVICE, CHAR, NAME
except Exception:
    SERVICE = '6e616461-6b69-7400-4b42-000000000001'
    CHAR = '6e616461-6b69-7400-4b42-000000000002'
    NAME = 'iverson'

try:
    from iverson.core import asset_dir, data_dir
except Exception:
    def asset_dir(): return Path(__file__).resolve().parent
    def data_dir():
        env = __import__('os').environ.get('IVERSON_DATA')
        if env: p = Path(env).expanduser(); p.mkdir(parents=True, exist_ok=True); return p
        p = Path.home() / '.local' / 'share' / 'iverson'; p.mkdir(parents=True, exist_ok=True); return p


## Number → glyph map


In [ ]:
#| export
CODES_WIN = dict(space=0x20, **{'return':0x0D}, tab=0x09, backspace=0x08, escape=0x1B,
                 left=0x25, up=0x26, right=0x27, down=0x28)
CODES_XD = dict(space='space', **{'return':'Return'}, tab='Tab', backspace='BackSpace', escape='Escape',
                left='Left', right='Right', up='Up', down='Down')
SP = {'space','return','tab','backspace','escape','left','right','up','down'}
_bad = {'Cc','Cf','Cn','Co','Cs'}

def _typable(t): return 0 < len(t) <= 8 and not any(c.isspace() or unicodedata.category(c) in _bad for c in t)
def _typed(sym): return sym.split(',')[0].replace('…','').strip() or sym

def _glyph_ename_map():
    "basedpl English name → glyph to type, from glyph_table.json."
    try:
        table = json.loads((asset_dir()/'glyph_table.json').read_text())
    except Exception:
        return {}
    out = {}
    for rows in table.values():
        if not isinstance(rows, list): continue
        for row in rows:
            if not isinstance(row, list) or len(row) < 1: continue
            sym = row[0]
            ename = row[4] if len(row) > 4 else ''
            if not ename: continue
            key = _typed(sym)
            if key: out[ename.strip().lower()] = key
    return out

_ename_cache = None

_ONES = {
    'zero':0,'oh':0,'o':0,'nought':0,
    'one':1,'two':2,'three':3,'four':4,'five':5,
    'six':6,'seven':7,'eight':8,'nine':9,
    'ten':10,'eleven':11,'twelve':12,'thirteen':13,'fourteen':14,
    'fifteen':15,'sixteen':16,'seventeen':17,'eighteen':18,'nineteen':19,
}
_TENS = {'twenty':20,'thirty':30,'forty':40,'fourty':40,'fifty':50,'sixty':60,'seventy':70,'eighty':80,'ninety':90}

def digitize(text):
    "Turn spoken number words into digit tokens. Mirrors Swift NumeralTranscript."
    parts = []
    for w in (text or '').lower().replace('-', ' ').replace(',', ' ').split():
        w = w.strip('.,;:!?' )
        if not w: continue
        if w.isdigit() or _is_float(w):
            parts.append(w); continue
        if w in _ONES and _ONES[w] < 10:
            parts.append(str(_ONES[w])); continue
        if w in _ONES:
            parts.append(str(_ONES[w])); continue
        if w in _TENS:
            parts.append(str(_TENS[w])); continue
        parts.append(w)
    return parts

def _is_float(s):
    try: float(s); return True
    except ValueError: return False

def name_map(db_path=None, force=False):
    "Map Sound Names numbers + basedpl English names → key string to type."
    global _ename_cache
    if _ename_cache is None or force:
        _ename_cache = _glyph_ename_map()
        log.info('loaded %d glyph-name mappings from glyph_table', len(_ename_cache))
    out = dict(_ename_cache)
    # Number→glyph library from Sound Names.
    path = Path(db_path) if db_path else data_dir()/'gnn.db'
    if path.is_file():
        try:
            from fastlite import database
            class Sound: id:int=None; name:str=''; glyph:str=''; key:str=''; samples:bytes=b''
            sounds = database(str(path)).create(Sound, pk='id')
            n = 0
            for s in sounds():
                if not s.name: continue
                k = s.key if s.key in SP or _typable(s.key or '') else (_typed(s.glyph) if s.glyph else '')
                if k:
                    out[s.name.strip()] = k
                    n += 1
            log.info('loaded %d number→glyph mappings from %s', n, path)
        except Exception as e:
            log.debug('sound db skipped: %s', e)
    return out

CODE_LEN = 3  # Sound Names codes are 3 digits (001–999)

def _sound_codes(toks):
    "Join digit tokens into full 3-digit codes (e.g. 0 0 2 → 002). Drop incomplete shorts."
    digits = ''.join(t for t in toks if t.isdigit())
    if not digits: return []
    if len(digits) < CODE_LEN:
        log.debug('ignore incomplete code %r (need %d digits)', digits, CODE_LEN)
        return []
    codes = []
    for i in range(0, len(digits), CODE_LEN):
        chunk = digits[i:i+CODE_LEN]
        if len(chunk) == CODE_LEN:
            codes.append(chunk)
        else:
            log.debug('ignore trailing incomplete digits %r', chunk)
    return codes

def resolve(text, mapping=None):
    "Turn iverson.io output into keys: numbers (Sound Names), glyphs, or basedpl names."
    mapping = name_map() if mapping is None else mapping
    raw = (text or '').strip()
    if not raw: return []

    # Digit / number-word tokens → one Sound Names code (join spaced digits).
    toks = digitize(raw)
    if toks and all(t.isdigit() or _is_float(t) for t in toks):
        keys = []
        for code in _sound_codes(toks):
            if code in mapping: keys.append(mapping[code])
            else: log.debug('no glyph for number %r', code)
        if keys: return keys
        codes = _sound_codes(toks)
        return codes if codes else []

    # Single known mapping key (number or special).
    if raw in SP or raw in mapping: return [mapping.get(raw, raw)]

    # Swift already mapped names → glyphs: type symbol / non-ASCII payload as-is.
    letters = sum(ch.isalpha() for ch in raw)
    if letters == 0 or any(ord(ch) > 127 for ch in raw):
        if any(ch in mapping.values() for ch in raw) or any(ord(ch) > 127 for ch in raw):
            return [raw]

    # Spoken English names ("plus", "up arrow", "less-than").
    tokens = raw.lower().replace('-', ' ').split()
    keys, i = [], 0
    phrases = sorted(
        ((tuple(n.replace('-', ' ').split()), g) for n, g in mapping.items() if any(c.isalpha() for c in n)),
        key=lambda x: len(x[0]), reverse=True)
    while i < len(tokens):
        hit = None
        for words, g in phrases:
            if not words: continue
            end = i + len(words)
            if end <= len(tokens) and tuple(tokens[i:end]) == words:
                hit = (len(words), g); break
        if hit:
            keys.append(hit[1]); i += hit[0]
        else:
            tok = tokens[i]
            if tok in mapping: keys.append(mapping[tok])
            else: log.debug('skip unknown token %r', tok)
            i += 1
    if not keys: log.warning('no glyphs for %r — speak a Sound Names number or basedpl name', raw[:80])
    return keys


## Inject keystrokes


In [ ]:
#| export
def _type_mac(k):
    from iverson import keys
    if k in keys.CODES: return keys.type_key(k)
    ok = True
    for ch in k:
        ok = keys.type_key(ch) and ok
    return ok

def _type_win(k):
    import ctypes
    from ctypes import wintypes
    user32 = ctypes.windll.user32
    INPUT_KEYBOARD, KEYEVENTF_KEYUP, KEYEVENTF_UNICODE = 1, 0x0002, 0x0004
    ULONG_PTR = ctypes.c_size_t
    class KEYBDINPUT(ctypes.Structure):
        _fields_ = [('wVk', wintypes.WORD), ('wScan', wintypes.WORD), ('dwFlags', wintypes.DWORD),
                    ('time', wintypes.DWORD), ('dwExtraInfo', ULONG_PTR)]
    class MOUSEINPUT(ctypes.Structure):
        _fields_ = [('dx', wintypes.LONG), ('dy', wintypes.LONG), ('mouseData', wintypes.DWORD),
                    ('dwFlags', wintypes.DWORD), ('time', wintypes.DWORD), ('dwExtraInfo', ULONG_PTR)]
    class HARDWAREINPUT(ctypes.Structure):
        _fields_ = [('uMsg', wintypes.DWORD), ('wParamL', wintypes.WORD), ('wParamH', wintypes.WORD)]
    class INPUTUNION(ctypes.Union):
        _fields_ = [('ki', KEYBDINPUT), ('mi', MOUSEINPUT), ('hi', HARDWAREINPUT)]
    class INPUT(ctypes.Structure):
        _fields_ = [('type', wintypes.DWORD), ('u', INPUTUNION)]
    def tap(vk=0, scan=0, flags=0):
        i = INPUT(type=INPUT_KEYBOARD, u=INPUTUNION(ki=KEYBDINPUT(vk, scan, flags, 0, 0)))
        user32.SendInput(1, ctypes.byref(i), ctypes.sizeof(i))
    if k in CODES_WIN:
        vk = CODES_WIN[k]; tap(vk=vk); tap(vk=vk, flags=KEYEVENTF_KEYUP); return True
    for ch in k:
        data = ch.encode('utf-16-le')
        for i in range(0, len(data), 2):
            scan = int.from_bytes(data[i:i+2], 'little')
            tap(scan=scan, flags=KEYEVENTF_UNICODE)
            tap(scan=scan, flags=KEYEVENTF_UNICODE | KEYEVENTF_KEYUP)
    return True

def _type_linux(k):
    import shutil, subprocess
    if not shutil.which('xdotool'): return False
    if k in CODES_XD: subprocess.run(['xdotool', 'key', CODES_XD[k]], check=False); return True
    subprocess.run(['xdotool', 'type', '--clearmodifiers', '--', k], check=False); return True

def inject(k):
    "Type `k` into the focused app on this machine. False when the platform won't allow it."
    t0 = time.perf_counter()
    try:
        if sys.platform == 'darwin': ok = _type_mac(k)
        elif sys.platform == 'win32': ok = _type_win(k)
        elif sys.platform.startswith('linux'): ok = _type_linux(k)
        else: ok = False
    except Exception:
        log.exception('inject raised for %r', k)
        return False
    ms = (time.perf_counter() - t0) * 1000
    log.info('inject %s %r (%d chars) in %.1fms platform=%s', 'ok' if ok else 'FAIL', k[:80], len(k), ms, sys.platform)
    return ok


## Scan and connect


In [ ]:
#| export
def _code_digits(code):
    return str(code).strip()

def _expected_name(code):
    "BLE local name for a 4-digit link code (`iverson-4821`)."
    return f'{NAME}-{_code_digits(code)}'.lower()

def _ad_has_code(ad, code):
    "True if advertisement manufacturer/service data carries the link code."
    digits = _code_digits(code).encode('ascii', errors='ignore')
    if not digits: return False
    for payload in (getattr(ad, 'manufacturer_data', None) or {}).values():
        if payload == digits or digits in payload: return True
    for payload in (getattr(ad, 'service_data', None) or {}).values():
        if payload == digits or digits in payload: return True
    return False

def _match(d, ad, code, nearby=None):
    "Match iverson link code. Prefer adv local_name — Bleak's d.name is often the phone GAP name."
    expected = _expected_name(code)
    digits = _code_digits(code)
    adv = (ad.local_name or '').lower()
    gap = (d.name or '').lower()
    names = {n for n in (adv, gap) if n}
    uuids = {u.lower() for u in (ad.service_uuids or [])}
    has_svc = SERVICE.lower() in uuids
    name_hit = expected in names or any(digits and digits in n and NAME in n for n in names)
    code_hit = _ad_has_code(ad, code)
    # Accept: exact/local name, or our service + manufacturer/service-data code.
    # Service UUID alone is not enough (avoids linking the wrong phone).
    hit = name_hit or (has_svc and code_hit)
    if nearby is not None and not hit:
        label = adv or gap
        if has_svc: nearby.add(f"{label or d.address} [iverson service]")
        elif label: nearby.add(label)
    if hit:
        log.info('scan HIT gap=%r local=%r addr=%s rssi=%s uuids=%s mfg=%s',
                 d.name, ad.local_name, d.address, getattr(ad, 'rssi', None), list(uuids)[:4],
                 dict(getattr(ad, 'manufacturer_data', None) or {}))
    else:
        log.debug('scan see gap=%r local=%r addr=%s rssi=%s uuids=%s mfg=%s',
                  d.name, ad.local_name, d.address, getattr(ad, 'rssi', None), list(uuids)[:2],
                  dict(getattr(ad, 'manufacturer_data', None) or {}))
    return hit

async def find(timeout=20.0, code=None):
    "Scan for an iverson.io peripheral advertising the given link code."
    if not _bleak: raise RuntimeError('Install bleak: pip install bleak')
    if not code: raise ValueError('link code required')
    expected = _expected_name(code)
    log.info('scanning for name=%r (or service+%s in mfg data) timeout=%.1fs', expected, _code_digits(code), timeout)
    t0 = time.perf_counter()
    nearby = set()
    def filt(d, ad): return _match(d, ad, code, nearby)
    dev = await BleakScanner.find_device_by_filter(filt, timeout=timeout)
    elapsed = time.perf_counter() - t0
    if dev:
        log.info('scan done in %.1fs found=%s', elapsed, getattr(dev, 'address', None))
    else:
        svc_hits = sorted(n for n in nearby if '[iverson service]' in n)
        phones = sorted(n for n in nearby if 'iphone' in n.lower() or 'ipad' in n.lower())
        others = sorted(nearby - set(phones) - set(svc_hits))[:6]
        if svc_hits:
            log.warning('scan done in %.1fs — saw iverson service on %s but link code %r did not match. Rebuild/run iverson.io, toggle Link, use the new code.',
                        elapsed, ', '.join(svc_hits), expected)
        elif phones:
            log.warning('scan done in %.1fs — saw %s but not %r (no iverson service in ads). Keep iverson.io foreground with Link on, then retry.',
                        elapsed, ', '.join(phones), expected)
        elif nearby:
            log.warning('scan done in %.1fs — no %r. Nearby named devices: %s', elapsed, expected, ', '.join(others))
        else:
            log.warning('scan done in %.1fs — no %r advertisement. Keep iverson.io open with Link on and check the code.', elapsed, expected)
    return dev

def resolve_char(client):
    "Pick our notify characteristic by service UUID, then handle (avoids Bleak duplicate-UUID errors)."
    svc_l, char_l = SERVICE.lower(), CHAR.lower()
    matches = []
    for svc in client.services or []:
        for ch in svc.characteristics:
            if ch.uuid.lower() != char_l: continue
            matches.append((svc.uuid.lower() == svc_l, ch))
            log.info('gatt candidate service=%s char=%s handle=%s props=%s ourservice=%s',
                     svc.uuid, ch.uuid, getattr(ch, 'handle', None), ch.properties, svc.uuid.lower() == svc_l)
    if not matches:
        log.error('no characteristic %s found', CHAR); return None
    # Prefer characteristic that lives under our service UUID.
    for ours, ch in matches:
        if ours and 'notify' in (ch.properties or []): return ch
    for ours, ch in matches:
        if ours: return ch
    for ours, ch in matches:
        if 'notify' in (ch.properties or []): return ch
    return matches[0][1]


## Run loop and CLI


In [ ]:
#| export
def _load_talk_key(db_path=None, override=None):
    if override: return override
    try:
        from iverson.main import talk_key
        return talk_key()
    except Exception:
        pass
    try:
        from fastlite import database
        path = Path(db_path) if db_path else data_dir()/'gnn.db'
        class Setting: key:str=''; value:str=''
        rows = database(str(path)).create(Setting, pk='key')
        row = next((x for x in rows() if x.key == 'talk_key'), None)
        from iverson.keys import normalize_talk_key
        return normalize_talk_key(getattr(row, 'value', None) if row else 'ctrl')
    except Exception:
        return 'ctrl'

async def run(timeout=20.0, db_path=None, code=None, talk=None, require_talk=True):
    "Connect, subscribe, map numbers→glyphs; type only while talk key is armed (unless disabled)."
    if not _bleak: raise SystemExit('Install bleak first: pip install bleak')
    if not code: raise SystemExit('Pass --code with the 4-digit number shown in iverson.io')
    name_map(db_path, force=True)  # log how many mappings we have at startup
    gate = None
    if require_talk:
        talk_name = _load_talk_key(db_path, talk)
        from iverson.keys import TalkGate, TALK_KEY_LABELS
        gate = TalkGate(talk_name)
        def flush(items):
            for text in items:
                keys_out = resolve(text, name_map(db_path))
                log.info('talk-flush text=%r → %s', text[:120], keys_out)
                for k in keys_out: inject(k)
        gate.set_flush(flush)
        if gate.start():
            label = dict(TALK_KEY_LABELS).get(gate.key, gate.key)
            log.info('talk key = %s — HOLD %s, speak, then release (armed %.0fs after release)',
                     gate.key, label, gate.grace)
        else:
            log.warning('talk-key unavailable — typing every notify')
            gate = None
    else:
        log.info('talk key disabled — typing every 3-digit code')
    n_connect, n_notify, n_disconnect = 0, 0, 0
    while True:
        dev = await find(timeout=timeout, code=code)
        if not dev:
            log.warning('no peripheral found — open iverson.io, tap Link, and pass the code with --code')
            await asyncio.sleep(2); continue
        n_connect += 1
        log.info('connecting #%d to name=%r addr=%s', n_connect, dev.name, dev.address)
        disconnected = asyncio.Event()
        def on_disconnect(client):
            nonlocal n_disconnect
            n_disconnect += 1
            log.warning('disconnect callback #%d connected=%s', n_disconnect, client.is_connected)
            disconnected.set()
        try:
            async with BleakClient(dev, disconnected_callback=on_disconnect, timeout=20.0) as client:
                svcs = list(client.services) if client.services else []
                log.info('connected mtu=%s services=%d', getattr(client, 'mtu_size', '?'), len(svcs))
                ch = resolve_char(client)
                if ch is None:
                    raise RuntimeError(f'characteristic {CHAR} not found under service {SERVICE}')
                handle = getattr(ch, 'handle', None)
                log.info('using char handle=%s uuid=%s props=%s', handle, ch.uuid, ch.properties)
                def on_notify(sender, data: bytearray):
                    nonlocal n_notify
                    n_notify += 1
                    text = data.decode('utf-8', errors='replace')
                    if gate is not None:
                        ok = gate.accept(text)
                        if ok:
                            log.info('notify #%d text=%r — typed (talk armed)', n_notify, text[:120])
                        else:
                            log.warning('notify #%d text=%r DROPPED — hold talk key, speak, release',
                                        n_notify, text[:120])
                        return
                    keys_out = resolve(text, name_map(db_path))
                    log.info('notify #%d text=%r → %s', n_notify, text[:120], keys_out)
                    for k in keys_out: inject(k)
                # Prefer handle to avoid “Multiple Characteristics with this UUID”.
                target = handle if handle is not None else ch
                log.info('start_notify target=%r', target)
                await client.start_notify(target, on_notify)
                log.info('subscribed — HOLD talk key, speak 3-digit code, release (focus a text field)')
                while client.is_connected and not disconnected.is_set():
                    await asyncio.sleep(0.5)
                log.warning('left connection loop is_connected=%s', client.is_connected)
        except asyncio.CancelledError:
            log.info('cancelled'); raise
        except Exception as e:
            n_disconnect += 1
            log.error('connection error #%d: %s: %s', n_disconnect, type(e).__name__, e)
            log.debug('traceback:\n%s', traceback.format_exc())
            await asyncio.sleep(1)
        log.info('totals connect=%d notify=%d disconnect=%d — reconnecting', n_connect, n_notify, n_disconnect)
        await asyncio.sleep(0.5)

def main(argv=None):
    "CLI entry: `iverson-receive --code XXXX`."
    p = argparse.ArgumentParser(description='Receive Sound Names numbers (or glyph names) from iverson.io over BLE and type glyphs.')
    p.add_argument('--code', required=True, help='4-digit link code shown in iverson.io when Link is on')
    p.add_argument('--talk-key', default=None, help='Talk key (ctrl, option, command, …). Default: Sound Keyboard setting')
    p.add_argument('--no-talk-key', action='store_true', help='Type every code immediately (disable talk-key gate)')
    p.add_argument('--timeout', type=float, default=20.0, help='Seconds to scan before retrying')
    p.add_argument('--db', default=None, help='Path to gnn.db (default: iverson data dir)')
    p.add_argument('-v', '--verbose', action='store_true', help='Log every advertisement seen while scanning')
    args = p.parse_args(argv)
    _setup_log(verbose=args.verbose)
    log.info('iverson-receive starting platform=%s bleak=%s service=%s code=%s', sys.platform, _bleak, SERVICE, args.code)
    try: asyncio.run(run(timeout=args.timeout, db_path=args.db, code=args.code, talk=args.talk_key, require_talk=not args.no_talk_key))
    except KeyboardInterrupt: log.info('stopped by user')

if __name__ == '__main__': main()



In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
